# Evaluation of the Metadata Annotations Module
This notebook contains the code for evaluation in section 5.2 Metadata Annotation Module of the *FairGround* paper. We show how the numbers in Table 2 is produced.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from prettytable import PrettyTable

In [ ]:
working_dir = Path("../experiments/face-eval")
automatic_annotations_path = working_dir / "fg/output/dataset_table.csv"
manual_annotations_path = "../manual_annotations/face/dataset_table.csv"

In [3]:
# we will save evaluation details in the evaluation directory
evaluation_dir = working_dir / "fg" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# load manual (ground truth) datasets and automatic datasets annotations
manual_df = pd.read_csv(manual_annotations_path)
automatic_df = pd.read_csv(automatic_annotations_path)

### Helper functions for evaluation

In [5]:
def eval_correct(auto, manual):
    return (auto == manual) \
           | (auto.isna() & manual.isna())

In [6]:
def eval_metrics(auto, manual, correct):
    n1 = (auto.notna() & manual.notna() & correct).sum()  # right answer
    n2 = (auto.notna() & manual.notna() & ~correct).sum()  # wrong answer
    n3 = (auto.isna() & manual.notna()).sum()  # missed
    n4 = (auto.notna() & manual.isna()).sum()  # hallucinated
    n5 = (auto.isna() & manual.isna()).sum()  # correctly abstained
    return n1, n2, n3, n4, n5

In [7]:
def print_abstension_confusion_matrix(counts):
    n1, n2, n3, n4, n5 = counts
    t = PrettyTable()
    t.field_names = ["True \\ Pred"] + ["Correctly\nanswered", "Incorrectly\nanswered", "Abstained"]
    t.add_row(["No abstention", n1, n2, n3])
    t.add_row(["Abstention", None, n4, n5])
    print(t)

In [8]:
def print_metrics(counts):
    n1, n2, n3, n4, n5 = counts
    r_acc = n1 / (n1 + n2 + n4) if (n1 + n2 + n4) else 0
    acc = (n1 + n5) / (n1 + n2 + n3 + n4 + n5)
    coverage = (n1 + n2) / (n1 + n2 + n3)
    print(f"Reliable Accuracy: {r_acc:.2f}")
    print(f"BAR: {coverage:.2f}")
    print(f"ACC: {acc:.2f}")

**Project Page URL:** Check if manual and automatic project URL page annotations are exact matches. 

In [9]:
correct = eval_correct(automatic_df["project_page"], manual_df["project_page"])
counts = eval_metrics(automatic_df["project_page"], manual_df["project_page"], correct)
print_abstension_confusion_matrix(counts)
print_metrics(counts)

+---------------+-----------+-------------+-----------+
|  True \ Pred  | Correctly | Incorrectly | Abstained |
+---------------+-----------+-------------+-----------+
| No abstention |     56    |      2      |     3     |
|   Abstention  |    None   |      0      |     5     |
+---------------+-----------+-------------+-----------+
Reliable Accuracy: 0.97
BAR: 0.95
ACC: 0.92


**Data License:** Check if manual and automatic data license annotations are exact matches. 

In [10]:
automatic_license = automatic_df["data_license"] \
                    .combine_first(automatic_df["website_data_license"])
manual_license = manual_df["data_license"] \
                 .combine_first(manual_df["website_data_license"])
license_correct = eval_correct(automatic_license, manual_license)
counts = eval_metrics(automatic_license, manual_license, license_correct)
print_abstension_confusion_matrix(counts)
print_metrics(counts)

+---------------+-----------+-------------+-----------+
|  True \ Pred  | Correctly | Incorrectly | Abstained |
+---------------+-----------+-------------+-----------+
| No abstention |     17    |      1      |     15    |
|   Abstention  |    None   |      0      |     33    |
+---------------+-----------+-------------+-----------+
Reliable Accuracy: 0.94
BAR: 0.55
ACC: 0.76


**Research Use Agreement:** Check if manual and automatic data license (supplemented by usage agreement if a license is not present) annotations are matches in terms of research use. Here, the matching process is manual. We manually read the license and usage agreement information extracted both manually and automatically, and  determine whether the two information agree on whether research use is allowed for the dataset. The manual evaluation results are saved in `{evaluation_dir}/agreement_manual.csv` and loaded here.

In [11]:
automatic_agreement = automatic_df["data_license"] \
                    .combine_first(automatic_df["website_data_license"]) \
                    .combine_first(automatic_df["website_usage_agreement"])
manual_agreement = manual_df["data_license"] \
                 .combine_first(manual_df["website_data_license"]) \
                 .combine_first(manual_df["website_usage_agreement"])
correct = pd.read_csv(evaluation_dir / "agreement_manual.csv")["correct"]
counts = eval_metrics(automatic_agreement, manual_agreement, correct)
print_abstension_confusion_matrix(counts)
print_metrics(counts)

+---------------+-----------+-------------+-----------+
|  True \ Pred  | Correctly | Incorrectly | Abstained |
+---------------+-----------+-------------+-----------+
| No abstention |     26    |      2      |     15    |
|   Abstention  |    None   |      0      |     23    |
+---------------+-----------+-------------+-----------+
Reliable Accuracy: 0.93
BAR: 0.65
ACC: 0.74
